# CLUTRR & RuleTaker: Kinship and Rule-Based Entailment Benchmarks

## Overview

This notebook demonstrates the dataset preparation pipeline for the pragmatic-tier annotation experiment. It processes two established NLP reasoning benchmarks:

- **CLUTRR (Compositional Language Understanding with Text-based Relational Reasoning)**: Kinship stories testing compositional generalization in relational reasoning. Published at EMNLP 2019.
- **RuleTaker**: Rule-based entailment examples spanning multiple reasoning depths. Published at IJCAI 2020.

Both datasets are standardized to a common schema with 'input' (story/context+question text), 'output' (relation label/entailment label), and metadata fields.

CLUTRR provides natural-language kinship stories where pragmatic tiers arise from explicit and implicit relational claims. RuleTaker provides rule+fact contexts where assertions, presuppositions, and entailments map directly onto pragmatic tier classification.

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# Always install these non-core packages
_pip('loguru==0.7.2')

# Core packages: pre-installed on Colab, install locally to match Colab environment
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'matplotlib==3.10.0')

In [ ]:
from pathlib import Path
import json
import re
import sys
from collections import defaultdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-09ca95-pragma-stratified-problog-grounding-hall/main/round-1/dataset-1/demo/mini_demo_data.json"
import json, os

def load_data():
    """Load data from GitHub URL with local fallback for development."""
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception as e:
        pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f:
            return json.load(f)
    raise FileNotFoundError(f"Could not load data from GitHub or local file")

In [ ]:
# Load the data
data = load_data()
print(f"Loaded data successfully: {len(data['datasets'])} datasets")

## Configuration

Set tunable parameters below. For the demo, we use minimal values that show the data structure and processing pipeline. For production runs, scale these up to the full dataset.

In [ ]:
# Tunable parameters for demo (minimal values)
MAX_EXAMPLES_PER_DATASET = 6  # Use all examples from each dataset in demo data
VERBOSE = True  # Print detailed logs

## Processing: CLUTRR Dataset

Process CLUTRR kinship classification examples. Each example contains:
- **input**: A story describing relationships between named characters
- **output**: A kinship relation label (integer code)
- **metadata_entity_pair**: The pair of entities being classified
- **metadata_task_type**: Always 'kinship_classification' for CLUTRR

In [ ]:
def process_clutrr(raw_data):
    """Extract and standardize CLUTRR examples."""
    clutrr_dataset = next((d for d in raw_data['datasets'] if d['dataset'] == 'clutrr'), None)
    if not clutrr_dataset:
        return []
    
    examples = clutrr_dataset['examples'][:MAX_EXAMPLES_PER_DATASET]
    
    if VERBOSE:
        print(f"\n=== CLUTRR Dataset ===")
        print(f"Loaded {len(examples)} examples")
        if examples:
            print(f"Sample input: {examples[0]['input'][:100]}...")
            print(f"Sample output: {examples[0]['output']}")
            print(f"Sample entity pair: {examples[0]['metadata_entity_pair']}")
    
    return examples

clutrr_examples = process_clutrr(data)

## Processing: RuleTaker Dataset

Process RuleTaker entailment classification examples. Each example contains:
- **input**: Context (facts and rules) plus a question claim
- **output**: Entailment label ('entailment' or 'not entailment')
- **metadata_config**: The reasoning depth configuration (NatLang, depth-1, etc.)
- **metadata_task_type**: Always 'entailment_classification' for RuleTaker

In [ ]:
def process_ruletaker(raw_data):
    """Extract and standardize RuleTaker examples."""
    ruletaker_dataset = next((d for d in raw_data['datasets'] if d['dataset'] == 'ruletaker'), None)
    if not ruletaker_dataset:
        return []
    
    examples = ruletaker_dataset['examples'][:MAX_EXAMPLES_PER_DATASET]
    
    if VERBOSE:
        print(f"\n=== RuleTaker Dataset ===")
        print(f"Loaded {len(examples)} examples")
        if examples:
            input_text = examples[0]['input']
            print(f"Sample input (first 150 chars): {input_text[:150]}...")
            print(f"Sample output: {examples[0]['output']}")
            print(f"Sample config: {examples[0]['metadata_config']}")
    
    return examples

ruletaker_examples = process_ruletaker(data)

## Results & Visualization

Display summary statistics, example samples, and metadata distribution.

In [ ]:
# Summary statistics
print("\n" + "="*60)
print("DEMO DATASET SUMMARY")
print("="*60)

total_examples = len(clutrr_examples) + len(ruletaker_examples)
print(f"\nTotal examples across both datasets: {total_examples}")
print(f"  CLUTRR (kinship):     {len(clutrr_examples)} examples")
print(f"  RuleTaker (entailment): {len(ruletaker_examples)} examples")

# Dataset-specific stats
if clutrr_examples:
    kinship_labels = [ex['output'] for ex in clutrr_examples]
    print(f"\nCLUTRR kinship labels (sample): {kinship_labels}")
    print(f"  Unique labels: {len(set(kinship_labels))}")

if ruletaker_examples:
    entailment_labels = [ex['output'] for ex in ruletaker_examples]
    label_counts = defaultdict(int)
    for label in entailment_labels:
        label_counts[label] += 1
    print(f"\nRuleTaker entailment labels:")
    for label, count in sorted(label_counts.items()):
        print(f"  {label}: {count}")
    
    configs = [ex['metadata_config'] for ex in ruletaker_examples]
    print(f"  Configs represented: {set(configs)}")

# Show metadata fields
print(f"\nMetadata description from artifact:")
print(data['metadata']['description'])
print(f"\nSource HuggingFace datasets:")
for hf_id in data['metadata']['source_hf_ids']:
    print(f"  - {hf_id}")

In [ ]:
# Display sample examples in readable format
print("\n" + "="*60)
print("SAMPLE EXAMPLES")
print("="*60)

if clutrr_examples:
    print("\n--- CLUTRR Example 1 ---")
    ex = clutrr_examples[0]
    print(f"Input: {ex['input']}")
    print(f"Output (kinship label): {ex['output']}")
    print(f"Entity pair: {ex['metadata_entity_pair']}")

if len(clutrr_examples) > 1:
    print("\n--- CLUTRR Example 2 ---")
    ex = clutrr_examples[1]
    print(f"Input: {ex['input']}")
    print(f"Output (kinship label): {ex['output']}")
    print(f"Entity pair: {ex['metadata_entity_pair']}")

if ruletaker_examples:
    print("\n--- RuleTaker Example 1 ---")
    ex = ruletaker_examples[0]
    print(f"Input (first 200 chars): {ex['input'][:200]}...")
    print(f"Output (entailment label): {ex['output']}")
    print(f"Config: {ex['metadata_config']}")

if len(ruletaker_examples) > 1:
    print("\n--- RuleTaker Example 2 ---")
    ex = ruletaker_examples[1]
    print(f"Input (first 200 chars): {ex['input'][:200]}...")
    print(f"Output (entailment label): {ex['output']}")
    print(f"Config: {ex['metadata_config']}")

In [ ]:
# Create visualization of dataset composition
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Pie chart: dataset distribution
sizes = [len(clutrr_examples), len(ruletaker_examples)]
labels = ['CLUTRR (kinship)', 'RuleTaker (entailment)']
colors = ['#ff9999', '#66b3ff']
axes[0].pie(sizes, labels=labels, colors=colors, autopct='%1.1f%%', startangle=90)
axes[0].set_title('Dataset Composition')

# Bar chart: entailment label distribution (RuleTaker only)
if ruletaker_examples:
    label_counts = defaultdict(int)
    for ex in ruletaker_examples:
        label_counts[ex['output']] += 1
    labels_rt = list(label_counts.keys())
    counts_rt = list(label_counts.values())
    axes[1].bar(labels_rt, counts_rt, color=['#90ee90', '#ffcc99'])
    axes[1].set_title('RuleTaker Entailment Labels')
    axes[1].set_ylabel('Count')
else:
    axes[1].text(0.5, 0.5, 'No RuleTaker examples', ha='center', va='center')
    axes[1].set_title('RuleTaker Label Distribution')

plt.tight_layout()
plt.savefig('/ai-inventor/aii_data/runs/5b0b4/4_gen_paper_repo/_3_gen_demo_art/notebook_workspaces/iter_1/art_Ze-oGbwxCQ3b/dataset_summary.png', dpi=100, bbox_inches='tight')
plt.show()

print("\nVisualization saved.")